In [ ]:
import requests
import csv
import time
from datetime import date, timedelta
 
 
START_DATE = date(2026, 4, 18)   
END_DATE   = date(2027, 4, 18) 
STEP_DAYS  = 1                  
                               
CURRENCY   = "USD"
OUTPUT_FILE = "cancun_hotel_prices.csv"
SLEEP_BETWEEN_REQUESTS = .8    
 
HOTELS = [
    {"name": "GR Solaris Cancun",                                   "key": "g150807-d501418"},
    {"name": "Dreams Sands Cancun Resort & Spa",                    "key": "g150807-d671540"},
    {"name": "Live Aqua Cancun",                                    "key": "g150807-d506374"},
    {"name": "Riu Palace Las Americas All Inclusive - Adults Only", "key": "g150807-d303140"},
    {"name": "Royalton CHIC Cancun, An Autograph Collection All-Inclusive Resort - Adults Only", "key": "g150807-d15580232"},
    {"name": "Grand Fiesta Americana Coral Beach",                  "key": "g150807-d152681"},
    {"name": "Moon Palace The Grand - Cancún",                      "key": "g150807-d13138532"},
    {"name": "Crown Paradise Club Cancun",                          "key": "g150807-d155814"},
    {"name": "Temptation Cancun Resort",                            "key": "g150807-d154428"},
    {"name": "Hotel Krystal Cancun",                                "key": "g150807-d154896"},
    {"name": "The Pyramid Cancun",                                  "key": "g150807-d8009670"},
    {"name": "Hilton Cancun, an All-Inclusive Resort",              "key": "g150807-d23146248"},
    {"name": "Paradisus Cancun",                                    "key": "g150807-d282106"},
    {"name": "Moon Palace Cancun",                                  "key": "g150807-d219163"},
    {"name": "Wyndham Grand Cancun All Inclusive Resort & Villas",  "key": "g150807-d154897"},
    {"name": "Kempinski Hotel Cancún",                              "key": "g150807-d152886"},
    {"name": "Hyatt Ziva Cancun",                                   "key": "g150807-d152887"},
    {"name": "Everglades Suites",                                   "key": "g150807-d33036291"},
    {"name": "The Sens Cancun",                                     "key": "g150807-d671741"},
    {"name": "SLS Cancun",                                          "key": "g150807-d19987759"},
    {"name": "Sandos Cancun",                                       "key": "g150807-d2554709"},
    {"name": "Krystal Grand Cancún All Inclusive",                  "key": "g150807-d155818"},
    {"name": "Waldorf Astoria Riviera Maya",                        "key": "g150807-d23540957"},
    {"name": "Sun Palace Cancun",                                   "key": "g150807-d152894"},
    {"name": "Secrets Mirabel Cancún Resort & Spa",                 "key": "g150807-d155958"},
    {"name": "GR Solaris Caribe",                                   "key": "g150807-d2034192"},
    {"name": "Grand Oasis Palm",                                    "key": "g150807-d566509"},
    {"name": "Dreams Vista Cancun Golf & Spa Resort",               "key": "g150807-d17765597"},
    {"name": "Breathless Cancun Soul Resort & Spa",                 "key": "g150807-d23760011"},
    {"name": "The Royal Sands All Suites Resort & Spa",             "key": "g150807-d153138"},
    {"name": "JW Marriott Cancun Resort & Spa",                     "key": "g150807-d209428"},
    {"name": "Secrets The Vine Cancun",                             "key": "g150807-d2627483"},
    {"name": "Le Blanc Spa Resort Cancun",                          "key": "g150807-d154868"},
    {"name": "Hard Rock Hotel Cancun",                              "key": "g150807-d152896"},
]
 
BASE_URL = "https://data.xotelo.com/api/rates"
 
 
def fetch_price(hotel_key: str, chk_in: str, chk_out: str) -> list[dict]:
    
    params = {
        "hotel_key": hotel_key,
        "chk_in":    chk_in,
        "chk_out":   chk_out,
        "currency":  CURRENCY,
        "adults":    2,
        "rooms":     1,
    }
    #Attempts to send a request to the API in this section and retrieve hotel rates.
    #If the request fails for any reason, an empty list is returned instead of crashing
    try:
        resp = requests.get(BASE_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        if data.get("error"):
            print(f"  API error: {data['error']}")
            return []
        return data.get("result", {}).get("rates", [])
    except Exception as e:
        print(f"  Request failed: {e}")
        return []
 
 
def main():
    rows = []
    total_requests = 0
    #Formats the string to set a check in and out date
    current = START_DATE
    while current <= END_DATE:
        chk_in  = current.strftime("%Y-%m-%d")
        chk_out = (current + timedelta(days=1)).strftime("%Y-%m-%d")
     #Loops through every Hotel in the list, printing a status message in
     #real time for progress insight. Calls the fetch_price function
    #with the hotel's key and current dates, storing whatever comes back.
        for hotel in HOTELS:
            print(f"Fetching {hotel['name']} | {chk_in} ...", end=" ")
            rates = fetch_price(hotel["key"], chk_in, chk_out)
            total_requests += 1
     #If no rates were found, prints no data, still following the same message
        #setup but with empty cells. 
            if not rates:
                print("no data")
                rows.append({
                    "date":       chk_in,
                    "hotel_name": hotel["name"],
                    "hotel_key":  hotel["key"],
                    "ota":        None,
                    "price_usd":  None,
                })
        #If rates were found, loops through each OTA, returning back onto the CSV
        #the typical provided information alongside the price from each OTA. 
            else:
                for ota in rates:
                    rows.append({
                        "date":       chk_in,
                        "hotel_name": hotel["name"],
                        "hotel_key":  hotel["key"],
                        "ota":        ota.get("name"),
                        "price_usd":  ota.get("rate"),
                    })
            #Grabs just the prices from the rows that were added, filtering None values. 
            #Prints the average price result to the line but gonna use a different
            #method in the main code block that deals with the actual csv provided by code. 
            #Average calculation kept to ensure accuracy in price matching. 
                prices = [r["price_usd"] for r in rows[-len(rates):] if r["price_usd"]]
                avg = round(sum(prices) / len(prices), 2) if prices else "N/A"
                print(f"{len(rates)} OTAs found | avg ${avg}")
             #Pauses for 0.8 seconds as requested by the actual Xotelo site as API server
            #to avoid the API from getting rate-limited or banned. 
            time.sleep(SLEEP_BETWEEN_REQUESTS)
 
        current += timedelta(days=STEP_DAYS)   #Moves onto next day
 
    #CSV
    #Typical CSV pasting information that displays all the informaiton on the CSV
    if rows:
        fieldnames = ["date", "hotel_name", "hotel_key", "ota", "price_usd"]
        with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"\nDone! {len(rows)} rows saved to '{OUTPUT_FILE}'")
        print(f"   Total API requests made: {total_requests}")
    else:
        print("\n No data collected.")
 
 
if __name__ == "__main__":
    main()
 